In [2]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 5.9 MB/s eta 0:00:0000:0100:01


In [3]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        endpoint = f'https://rest.uniprot.org/uniprotkb/{accession}.json'
        http_args = {'headers': {'Accept': 'application/json'}}
        return requests.get(endpoint, **http_args)

    def _get_ensembl(self, id):
        endpoint = f'https://rest.ensembl.org/lookup/id/{id}'
        http_args = {'headers': {'Accept': 'application/json'}}
        return requests.get(endpoint, **http_args)

    def _uniprot_parse_response(self, resp):
        try:
            data = resp.json()

            if resp.status_code != 200:
                acc = resp.url.split('/')[-1].replace('.json', '')
                if 'messages' in data:
                    return {acc: 'error:' + '; '.join(data['messages'])}
                return {acc: 'error:uniprot request failed'}

            acc = data['primaryAccession']
            output = {
                acc: {
                    'organism': data['organism']['scientificName'],
                    'geneInfo': data['genes'],
                    'sequenceInfo': data['sequence'],
                    'type': 'protein'
                }
            }
            return output

        except Exception:
            try:
                acc = resp.url.split('/')[-1].replace('.json', '')
                return {acc: 'error:failed to parse uniprot response'}
            except Exception:
                return {'unknown': 'error:failed to parse uniprot response'}

    def _ensembl_parse_response(self, resp):
        try:
            data = resp.json()

            if resp.status_code != 200:
                ens_id = resp.url.split('/')[-1]
                if 'error' in data:
                    return {ens_id: 'error:' + data['error']}
                return {ens_id: 'error:ensembl request failed'}

            ens_id = data['id']
            output = {
                ens_id: {
                    'object_type': data['object_type'],
                    'species': data['species'],
                    'assembly_name': data['assembly_name'],
                    'biotype': data['biotype'],
                    'display_name': data['display_name'],
                    'id': data['id'],
                    'db_type': data['db_type'],
                    'description': data['description'],
                    'source': data['source'],
                    'canonical_transcript': data['canonical_transcript']
                }
            }
            return output

        except Exception:
            try:
                ens_id = resp.url.split('/')[-1]
                return {ens_id: 'error:failed to parse ensembl response'}
            except Exception:
                return {'unknown': 'error:failed to parse ensembl response'}

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        if database == 'uniprot':
            db_response = self._get_uniprot(id)
            parsed_db = self._uniprot_parse_response(db_response)
        elif database == 'ensembl':
            db_response = self._get_ensembl(id)
            parsed_db = self._ensembl_parse_response(db_response)
        else:
            return {
                'DB_name': 'unknown',
                f'file_info_{id}': {
                    'description': seq_description,
                    'sequence': seq_sequence
                },
                f'database_info_{id}': {'error': 'unknown database'}
            }

        return {
            'DB_name': database,
            f'file_info_{id}': {
                'description': seq_description,
                'sequence': seq_sequence
            },
            f'database_info_{id}': parsed_db.get(id, parsed_db)
        }

    def seqkit_stats(self) -> dict:
        try:
            result = subprocess.run(
                ['seqkit', 'stats', self.filename],
                capture_output=True,
                text=True,
                check=True
            )
        except subprocess.CalledProcessError as e:
            return {'ERROR': e.stderr.strip()}
        except FileNotFoundError:
            return {'ERROR': 'seqkit is not installed or not found in PATH'}

        lines = result.stdout.strip().split('\n')
        if len(lines) < 2:
            return {'ERROR': 'Unexpected seqkit output'}

        header = lines[0].split()
        values = lines[1].split()

        if len(header) != len(values):
            return {'ERROR': 'Could not parse seqkit stats output correctly'}

        stat_info = dict(zip(header, values))

        fasta_type = stat_info.get('type')
        fasta_num_seqs = stat_info.get('num_seqs')

        try:
            fasta_num_seqs = int(fasta_num_seqs)
        except Exception:
            pass

        return {
            'fasta_seqkit_stat_info': stat_info,
            'fasta_type': fasta_type,
            'fasta_num_seqs': fasta_num_seqs
        }

    def biopython_parser(self, seqkit_result) -> dict:
        if 'ERROR' in seqkit_result:
            return seqkit_result

        fasta_type = seqkit_result.get('fasta_type')

        if fasta_type == 'Protein':
            database = 'uniprot'
            id_pattern = re.compile(r'\b(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9][A-Z0-9]{3}[0-9])\b')
        elif fasta_type in ['DNA', 'RNA']:
            database = 'ensembl'
            id_pattern = re.compile(r'\bENS[A-Z0-9]*\d+\b')
        else:
            return {'ERROR': f'Unsupported FASTA type: {fasta_type}'}

        output = {'DB_name': database}
        warning_counter = 1

        try:
            for record in SeqIO.parse(self.filename, 'fasta'):
                description = record.description
                sequence = str(record.seq)

                match = id_pattern.search(description)

                if match:
                    found_id = match.group()
                    seq_result = self._access_database(
                        found_id,
                        database,
                        description,
                        sequence
                    )

                    for key, value in seq_result.items():
                        if key == 'DB_name':
                            continue
                        output[key] = value
                else:
                    output[f'WARNING_{warning_counter}'] = {'No ID match found.': description}
                    warning_counter += 1

            return output

        except Exception as e:
            return {'ERROR': f'Biopython parsing failed: {str(e)}'}

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [6]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'file': 'test_file.fasta',
  'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228',
  'max_len': '427'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [5]:
biopython = parser.biopython_parser(stats)

parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

In [7]:
for fname in ['ensembl_download_1.fasta', 'ensembl_download_2.fasta', 'uniprot_download.fasta']:
    print('=' * 80)
    print(fname)
    parser = MyFastaParser(fname)
    stats = parser.seqkit_stats()
    print(stats)
    result = parser.biopython_parser(stats)
    parser.show_output(result)
    print()

ensembl_download_1.fasta
{'fasta_seqkit_stat_info': {'file': 'ensembl_download_1.fasta', 'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23'}, 'fasta_type': 'DNA', 'fasta_num_seqs': 6}
DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	error:failed to parse ensembl response
file_info_ENSMUST00000177564
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
database_info_ENSMUST0000017756